# 🌊 Bronze Streaming Ingest (Auto Loader)

Replaces `bronze_delta_ingest.ipynb`'s batch CSV-glob read with **Auto
Loader**, incrementally discovering only new files via a checkpoint.

**Why this is scoped as incremental file ingestion, not true streaming:**
the source PDFs have no natural event-time column and this is a fixed,
one-time 2025 dataset — there's no ongoing feed of new CAP-round PDFs
arriving in production. What Auto Loader still gives us that a plain batch
read doesn't: **exactly-once file discovery via checkpoint** (re-running
never reprocesses a file it's already ingested) and **schema
inference/evolution tracking** across runs. We prove the mechanism works
with a simulated test (Cell 5) rather than relying on the dataset ever
genuinely changing.

**Source:** same CSVs as `bronze_delta_ingest.ipynb`
(`/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/data/mahacet_cutoffs_2025_csv`)
**Output:** `rankrangers_project_data.bronze.mhcet_allotments_raw` (same
table — this notebook is a drop-in replacement for how it gets populated).

**Mode change vs. `bronze_delta_ingest.ipynb`:** that notebook does a full
`overwrite` every run (re-reads and replaces everything). This notebook
`append`s only newly-discovered files — the checkpoint is what makes that
safe (no risk of double-counting a file across runs).

## Cell 1 — Config

In [ ]:
import uuid

CSV_ROOT   = '/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/data/mahacet_cutoffs_2025_csv'
CATALOG    = 'rankrangers_project_data'
SCHEMA     = 'bronze'
TABLE      = 'mhcet_allotments_raw'
FULL_TABLE = f'{CATALOG}.{SCHEMA}.{TABLE}'

# Auto Loader needs a persistent location to track discovered files (the
# checkpoint) and the schema it inferred on first run (schemaLocation) -
# both live on the same Volume as the source CSVs so no new Volume/mount
# is needed.
CHECKPOINT_LOCATION = '/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/_checkpoints/bronze_autoloader/checkpoint'
SCHEMA_LOCATION      = '/Volumes/rankrangers_project_data/pdf/cet_raw_pdfs/_checkpoints/bronze_autoloader/schema'

BATCH_ID = str(uuid.uuid4())

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')
print(f'Source     : {CSV_ROOT}')
print(f'Target     : {FULL_TABLE}')
print(f'Checkpoint : {CHECKPOINT_LOCATION}')
print(f'Batch ID   : {BATCH_ID}')

## Cell 2 — Auto Loader Stream: New Files Since Last Checkpoint

`trigger(availableNow=True)` makes this a **triggered incremental batch**,
not an always-on stream — it processes whatever's new right now, then
stops. This is the right trigger for a scheduled Job task (Phase 6): it
runs to completion so downstream tasks can safely depend on it, instead of
running forever like a continuous stream would.

In [ ]:
from pyspark.sql import functions as F

stream_df = (spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format', 'csv')
    .option('cloudFiles.inferColumnTypes', True)
    .option('cloudFiles.schemaLocation', SCHEMA_LOCATION)
    .option('header', True)
    .option('encoding', 'UTF-8')
    .load(CSV_ROOT)
    .withColumn('_source_file', F.col('_metadata.file_path'))
    .withColumn('_ingest_ts', F.current_timestamp())
    .withColumn('_batch_id', F.lit(BATCH_ID)))

query = (stream_df.writeStream
    .format('delta')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .option('mergeSchema', 'true')
    .outputMode('append')
    .trigger(availableNow=True)
    .toTable(FULL_TABLE))

query.awaitTermination()
print('✅ Stream batch complete')

## Cell 3 — Verify: Rows Ingested This Run

In [ ]:
this_run = spark.table(FULL_TABLE).filter(F.col('_batch_id') == BATCH_ID)
new_rows  = this_run.count()
new_files = this_run.select('_source_file').distinct().count()
total_rows = spark.table(FULL_TABLE).count()

print(f'New rows ingested this run : {new_rows:,}')
print(f'New files ingested this run: {new_files:,}')
print(f'Total rows in {FULL_TABLE}: {total_rows:,}')

## Cell 4 — Sanity Check (first run only)

On a completely fresh table, this run should have ingested everything -
same 735,136-row / 1,480-file baseline `bronze_delta_ingest.ipynb` verifies.

In [ ]:
EXPECTED_RAW_ROWS  = 735_136
EXPECTED_RAW_FILES = 1_480

if total_rows == EXPECTED_RAW_ROWS:
    print(f'✅ Total row count matches the {EXPECTED_RAW_ROWS:,} baseline (first run).')
else:
    print(f'ℹ️  Total rows ({total_rows:,}) != {EXPECTED_RAW_ROWS:,} baseline - '
          f'expected on any run after the first, since Auto Loader only '
          f'appends newly-discovered files rather than re-reading everything.')

## Cell 5 — Prove the Checkpoint Works (manual simulated-arrival test)

Run Cells 1-3 once first so a checkpoint exists. Then, **outside this
notebook**, copy one existing CSV to a new filename in the same Volume
folder to simulate a new file arriving, e.g. via a notebook cell:
```python
dbutils.fs.cp(
    f'{CSV_ROOT}/CAP-I/CAPR-I_01002.csv',
    f'{CSV_ROOT}/CAP-I/CAPR-I_01002_simulated_new_arrival.csv'
)
```
Then re-run Cells 1-3. Expected result: `new_files` == 1 and `new_rows`
equals exactly that one file's row count - proving the checkpoint picked
up only the new file, not the other 1,480 already-ingested ones.

**Cleanup after the test:**
```python
dbutils.fs.rm(f'{CSV_ROOT}/CAP-I/CAPR-I_01002_simulated_new_arrival.csv')
```
(The extra rows stay in the Bronze table tagged with their `_batch_id` /
`_source_file` - harmless duplicates of real data, but delete them via
`DELETE FROM {FULL_TABLE} WHERE _source_file LIKE '%simulated_new_arrival%'`
before treating this table as production-clean again.)